In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════════
DITU-net — Training + Inference Pipeline  (X = GP (actual) / Y = TP (capability))
═══════════════════════════════════════════════════════════════════════════════
One file, cell-by-cell. Run each "# CELL" block in order on Kaggle GPU.

DESIGN (deployment-driven, corrected)
  At deployment the operator feeds ACTUAL grid power (GP). So the model is
  TRAINED on GP scatter and INFERS on GP scatter -> train and test match.
      X = (wind speed, actual grid power GP) scatter   [what we deploy on]
      Y = clean S-curve fitted from theoretical power TP [true capability]
  The model learns: messy actual output -> true capability curve. Curtailment
  appears in X (as it will at deployment); the model is made robust to it via
  heavy synthetic curtailment-shaped diversity in the dataset generator.

WHAT THIS FILE DOES
  1. GROUP-BY-MONTH split (GroupShuffleSplit + GroupKFold) using groups.json:
     all augmentations of a turbine-month stay on the SAME side of every split
     -> honest validation/test (no augmentation leakage).
  2. Fixed curve extractor (image_to_curve): NaN-masking, mean-of-argmin,
     monotone guard, C0-continuous flats -> removes the step/vertical-riser
     artefact. This is the base paper's Sec. III-B pixel-map + DKC, ported.
  3. 5-fold CV x 3 seeds = 15 runs (mean±std) + seed sweep 42..45 bonus.
  4. Paper-metric reporting (RMSE/MAE/MAPE/WMAPE/alpha-SS) for S1/S2/S3,
     directly comparable to the base paper Table IV.
  5. Full reproducibility: split_indices.npz, run_manifest.json, deploy ckpt.

INFERENCE INPUT SOURCE
  INFER_SOURCE = 'GP' (default) — matches training and deployment. Evaluation
  reference is always the TP 90th-percentile bins (independent of the model),
  so no curve is fit to the scoring target.
═══════════════════════════════════════════════════════════════════════════════
"""

# ═══════════════════════════════════════════════════════════════
# CELL 1 — Imports, Config, Reproducibility
# ═══════════════════════════════════════════════════════════════
import os, io, json, time, random, gc
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, KFold

# ── Paths (adjust to your uploaded dataset) ──────────────────────────────────
# The generator produced: training images (trainNx.jpg / trainNy.jpg),
# turbine_norms.json and groups.json. Upload them as a Kaggle dataset and
# point DATA_DIR at the image folder. The resolver below also checks
# /kaggle/working in case you run generation + training in one kernel.
DATA_DIR = '/kaggle/input/datasets/hamzatauqeernust/act-2-hybrid-5800-final'

CONSOLIDATED_CSV = ('/kaggle/input/datasets/hamzatauqeernust/act2-consolidated3yrs-csv/consolidated_wind_data_2023_2025.csv')

# turbine_norms.json / groups.json candidate locations (first existing wins)
NORM_CANDIDATES = [
    os.path.join(DATA_DIR, 'turbine_norms.json'),
    '/kaggle/input/datasets/hamzatauqeernust/act-2-norms-latest/turbine_norms.json',
    '/kaggle/working/turbine_norms.json',
]
GROUP_CANDIDATES = [
    os.path.join(DATA_DIR, 'groups.json'),
    '/kaggle/input/datasets/hamzatauqeernust/act-2-norms-latest/groups.json',
]

SAVE_DIR = '/kaggle/working/model'
RES_DIR  = '/kaggle/working/results'
CV_DIR   = '/kaggle/working/cv_runs'
for d in (SAVE_DIR, RES_DIR, CV_DIR):
    os.makedirs(d, exist_ok=True)

# ── Hyperparameters ──────────────────────────────────────────────────────────
NUM_SAMPLES  = 7000      # upper bound of pair indices to scan (>= total pairs)
BATCH_SIZE   = 16
NUM_EPOCHS   = 30
LR           = 1e-3
DROPOUT      = 0.10
WEIGHT_DECAY = 1e-5
PATIENCE     = 7         # early-stopping patience (on val MSE)
LR_PATIENCE  = 4         # ReduceLROnPlateau patience

# ── Split design (Andrew-Ng style: held-out test never touched by CV) ────────
TEST_FRACTION = 0.15     # final held-out test set (GROUP-disjoint)
N_FOLDS       = 5        # K-Fold over the remaining 85% dev set (GROUP-disjoint)
SEEDS         = [42, 43, 44]   # 3 seeds per fold -> 5*3 = 15 runs
SPLIT_SEED    = 2024     # fixed seed for the test hold-out (keep constant!)

# ── Inference input source (see header note) ─────────────────────────────────
INFER_SOURCE = 'GP'      # 'GP' = matches training + deployment (correct) | 'TP'

MODEL_NAME = 'ACT2'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def _resolve(cands):
    """Return first existing path from a candidate list, else None."""
    for p in cands:
        if p and os.path.exists(p):
            return p
    return None


def set_seed(seed: int, deterministic: bool = True):
    """Seed every RNG for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = True


def seed_worker(worker_id):
    """Make DataLoader workers deterministic."""
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)


print(f"Device       : {device}")
print(f"Data dir     : {DATA_DIR}")
print(f"Infer source : {INFER_SOURCE}  (GP=deployment-consistent)")
print(f"CV plan      : {N_FOLDS}-fold x {len(SEEDS)} seeds = {N_FOLDS*len(SEEDS)} runs")
print("✅ Cell 1 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Dataset + GROUP-aware Split Plan (no augmentation leakage)
# ═══════════════════════════════════════════════════════════════
# INPUT  X : RGB 256x256 scatter (TP)        -> in_channels=3
# OUTPUT Y : grayscale 256x256 S-curve       -> out_channels=1
#
# Split design (group = turbine-month for real pairs, unique id for synthetic):
#   1. Hold out a GROUP-disjoint TEST set (15%) with a FIXED seed.
#   2. On the remaining DEV set (85%), GroupKFold -> train/val per fold.
#   Because we split by GROUP, all 5 augmentations of a month land together,
#   so the model never validates/tests on a month it partly trained on.

class WPCDataset(Dataset):
    def __init__(self, data_dir, num_samples, input_transform=None, target_transform=None):
        self.data_dir = data_dir
        self.input_transform = input_transform
        self.target_transform = target_transform
        self.valid = []
        for i in range(num_samples):
            for ext in ('jpg', 'png'):
                xp = os.path.join(data_dir, f'train{i}x.{ext}')
                yp = os.path.join(data_dir, f'train{i}y.{ext}')
                if os.path.exists(xp) and os.path.exists(yp):
                    self.valid.append((i, ext)); break
        print(f"Found {len(self.valid)} valid pairs (scanned {num_samples}).")

    def __len__(self):
        return len(self.valid)

    def __getitem__(self, idx):
        i, ext = self.valid[idx]
        x = Image.open(os.path.join(self.data_dir, f'train{i}x.{ext}')).convert('RGB')
        y = Image.open(os.path.join(self.data_dir, f'train{i}y.{ext}')).convert('L')
        if self.input_transform:  x = self.input_transform(x)
        if self.target_transform: y = self.target_transform(y)
        return x, y


# Input gets light jitter (robustness); target must stay clean (no jitter).
input_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
target_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

full_ds = WPCDataset(DATA_DIR, NUM_SAMPLES, input_transform, target_transform)
N = len(full_ds)
DATASET_LEN = N
positions = np.arange(N)

# ── Build the group label for each dataset position ──────────────────────────
GROUP_PATH = _resolve(GROUP_CANDIDATES)
if GROUP_PATH is not None:
    with open(GROUP_PATH) as fh:
        _gmap = json.load(fh)   # {pair_idx(str): group_id}
    group_arr = np.array([_gmap.get(str(full_ds.valid[p][0]), f'unk_{p}')
                          for p in positions])
    GROUP_MODE = 'group'
    print(f"groups.json loaded ({GROUP_PATH}) — {len(set(group_arr))} unique groups")
else:
    # Fallback: per-image grouping (each image its own group). This reduces to a
    # plain split. WARNING: if you used TIER1_AUG>1, this can leak — regenerate
    # with groups.json to be safe.
    group_arr = np.array([f'img_{p}' for p in positions])
    GROUP_MODE = 'per_image'
    print("⚠️ groups.json NOT found — falling back to per-image split (possible leak "
          "if augmentation was used). Regenerate dataset with groups.json.")

# ── 1. Held-out TEST split (GROUP-disjoint, fixed) ───────────────────────────
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=SPLIT_SEED)
dev_pos, test_pos = next(gss.split(positions, groups=group_arr))
dev_pos, test_pos = np.sort(dev_pos), np.sort(test_pos)
print(f"Total {N} | dev {len(dev_pos)} | test {len(test_pos)} (group-disjoint, fixed)")

# ── 2. GroupKFold over DEV positions (group-disjoint train/val) ──────────────
dev_groups = group_arr[dev_pos]
gkf = GroupKFold(n_splits=N_FOLDS)
folds = []
for tr, va in gkf.split(dev_pos, groups=dev_groups):
    folds.append((dev_pos[tr], dev_pos[va]))
for f, (tr, va) in enumerate(folds):
    # sanity: assert no group overlap between train and val
    overlap = set(group_arr[tr]) & set(group_arr[va])
    print(f"  Fold {f}: train={len(tr):4d}  val={len(va):4d}  group_overlap={len(overlap)}")

# Persist split for reproducibility.
np.savez(os.path.join(SAVE_DIR, 'split_indices.npz'),
         test_pos=test_pos, dev_pos=dev_pos, group_mode=np.array([GROUP_MODE]),
         **{f'fold{f}_train': folds[f][0] for f in range(N_FOLDS)},
         **{f'fold{f}_val':   folds[f][1] for f in range(N_FOLDS)})
print("✅ Cell 2 done — group-aware split saved to split_indices.npz")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — U-Net (out_channels=1, dropout, batchnorm)
# ═══════════════════════════════════════════════════════════════
# Differs from base paper only by: grayscale output + Dropout2d in encoder.
# Skip connections concatenate encoder/decoder features (core of U-Net).

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, init_features=32, dropout=0.0):
        super().__init__()
        f = init_features
        self.drop = nn.Dropout2d(p=dropout)
        self.enc1 = self._block(in_channels, f,   'enc1')
        self.enc2 = self._block(f,           f*2, 'enc2')
        self.enc3 = self._block(f*2,         f*4, 'enc3')
        self.enc4 = self._block(f*4,         f*8, 'enc4')
        self.pool = nn.MaxPool2d(2, 2)
        self.bottleneck = self._block(f*8, f*16, 'btn')
        self.up4 = nn.ConvTranspose2d(f*16, f*8, 2, 2); self.dec4 = self._block(f*16, f*8, 'dec4')
        self.up3 = nn.ConvTranspose2d(f*8,  f*4, 2, 2); self.dec3 = self._block(f*8,  f*4, 'dec3')
        self.up2 = nn.ConvTranspose2d(f*4,  f*2, 2, 2); self.dec2 = self._block(f*4,  f*2, 'dec2')
        self.up1 = nn.ConvTranspose2d(f*2,  f,   2, 2); self.dec1 = self._block(f*2,  f,   'dec1')
        self.out_conv = nn.Conv2d(f, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(self.drop(e1)))
        e3 = self.enc3(self.pool(self.drop(e2)))
        e4 = self.enc4(self.pool(self.drop(e3)))
        b  = self.bottleneck(self.pool(self.drop(e4)))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.out_conv(d1)

    @staticmethod
    def _block(in_ch, feat, name):
        return nn.Sequential(OrderedDict([
            (f'{name}_c1', nn.Conv2d(in_ch, feat, 3, padding=1, bias=False)),
            (f'{name}_n1', nn.BatchNorm2d(feat)),
            (f'{name}_r1', nn.ReLU(inplace=True)),
            (f'{name}_c2', nn.Conv2d(feat, feat, 3, padding=1, bias=False)),
            (f'{name}_n2', nn.BatchNorm2d(feat)),
            (f'{name}_r2', nn.ReLU(inplace=True)),
        ]))


_tmp = UNet(3, 1, 32, DROPOUT)
print(f"Trainable params: {sum(p.numel() for p in _tmp.parameters() if p.requires_grad):,}")
del _tmp
print("✅ Cell 3 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — train_one_run(): trains ONE fold with ONE seed
# ═══════════════════════════════════════════════════════════════
# Saves only the BEST checkpoint per run (15 runs x 20 epochs).
# Prints live epoch updates every epoch so you can monitor progress.

def rmse_to_pixacc(mse):
    """Pixel-MSE proxy 'accuracy'. NOTE: NOT the paper's similarity accuracy."""
    return max(0.0, (1.0 - mse**0.5) * 100.0)


def train_one_run(train_index, val_index, seed, fold, epochs=NUM_EPOCHS, verbose=True):
    set_seed(seed)
    g = torch.Generator(); g.manual_seed(seed)

    train_ds = Subset(full_ds, list(train_index))
    val_ds   = Subset(full_ds, list(val_index))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True,
                              worker_init_fn=seed_worker, generator=g)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    model     = UNet(3, 1, 32, DROPOUT).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', factor=0.5, patience=LR_PATIENCE)

    tag = f'fold{fold}_seed{seed}'
    best_val, best_epoch, no_improve = float('inf'), -1, 0
    tr_hist, va_hist = [], []
    ckpt_path = os.path.join(CV_DIR, f'Unet_{MODEL_NAME}_{tag}_best.pth')

    # ── header line for this run ──────────────────────────────────────────
    print(f"\n{'─'*65}")
    print(f"  RUN : {tag}  |  train={len(train_ds)}  val={len(val_ds)}  "
          f"bs={BATCH_SIZE}  epochs={epochs}")
    print(f"{'─'*65}")
    print(f"  {'Ep':>3}  {'Train MSE':>10}  {'Val MSE':>10}  "
          f"{'PixAcc%':>8}  {'LR':>9}  {'Status'}")
    print(f"  {'─'*3}  {'─'*10}  {'─'*10}  {'─'*8}  {'─'*9}  {'─'*14}")

    run_start = time.time()

    for epoch in range(1, epochs + 1):
        ep_start = time.time()

        # ── train ─────────────────────────────────────────────────────────
        model.train(); tr_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward(); optimizer.step()
            tr_loss += loss.item()
        tr_loss /= len(train_loader)

        # ── validate ──────────────────────────────────────────────────────
        model.eval(); va_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                va_loss += criterion(model(xb), yb).item()
        va_loss /= len(val_loader)

        scheduler.step(va_loss)
        tr_hist.append(tr_loss); va_hist.append(va_loss)

        cur_lr   = optimizer.param_groups[0]['lr']
        pix_acc  = rmse_to_pixacc(va_loss)
        ep_time  = time.time() - ep_start

        # ── checkpoint logic ──────────────────────────────────────────────
        if va_loss < best_val - 1e-6:
            best_val, best_epoch, no_improve = va_loss, epoch, 0
            torch.save({'epoch': epoch, 'model': model.state_dict(),
                        'val_loss': va_loss, 'train_loss': tr_loss,
                        'fold': fold, 'seed': seed}, ckpt_path)
            status = f"✅ best saved"
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                status = f"🛑 early stop"
            elif no_improve >= PATIENCE - 1:
                status = f"⚠️  patience {no_improve}/{PATIENCE}"
            else:
                status = f"   patience {no_improve}/{PATIENCE}"

        # ── live epoch print (every epoch) ────────────────────────────────
        print(f"  {epoch:>3}  {tr_loss:>10.5f}  {va_loss:>10.5f}  "
              f"{pix_acc:>7.2f}%  {cur_lr:>9.6f}  {status}  "
              f"[{ep_time:.0f}s]")

        if no_improve >= PATIENCE:
            break

    # ── run summary ───────────────────────────────────────────────────────
    total_time = time.time() - run_start
    print(f"{'─'*65}")
    print(f"  DONE  best_val={best_val:.5f}  "
          f"pixacc={rmse_to_pixacc(best_val):.2f}%  "
          f"@ ep{best_epoch}  total={total_time/60:.1f} min")
    print(f"  ckpt → {ckpt_path}")
    print(f"{'─'*65}\n")

    del model, optimizer
    gc.collect(); torch.cuda.empty_cache()

    return {'fold': fold, 'seed': seed,
            'best_val_mse': best_val, 'best_epoch': best_epoch,
            'pix_acc': rmse_to_pixacc(best_val), 'ckpt': ckpt_path,
            'train_hist': tr_hist, 'val_hist': va_hist}

print("✅ Cell 4 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — 5-Fold CV x 3 Seeds = 15 runs  → mean ± std
# ═══════════════════════════════════════════════════════════════
# K-Fold measures split/data variance; the 3 seeds per fold isolate weight-init
# variance. Report mean ± std over all 15 as the headline robustness number.

all_runs, t_start = [], time.time()
for fold, (tr_idx, va_idx) in enumerate(folds):
    for seed in SEEDS:
        print(f"▶ Fold {fold} | Seed {seed}")
        all_runs.append(train_one_run(tr_idx, va_idx, seed, fold))
print(f"\nAll {len(all_runs)} runs done in {(time.time()-t_start)/60:.1f} min")

df_runs = pd.DataFrame([{k: r[k] for k in
                         ('fold', 'seed', 'best_val_mse', 'best_epoch', 'pix_acc', 'ckpt')}
                        for r in all_runs])
df_runs.to_csv(os.path.join(RES_DIR, 'cv_all_runs.csv'), index=False)

per_fold = df_runs.groupby('fold').agg(
    val_mse_mean=('best_val_mse', 'mean'), val_mse_std=('best_val_mse', 'std'),
    pixacc_mean=('pix_acc', 'mean'), pixacc_std=('pix_acc', 'std')).round(5)
per_fold.to_csv(os.path.join(RES_DIR, 'cv_per_fold.csv'))

overall = {'val_mse_mean': df_runs['best_val_mse'].mean(),
           'val_mse_std':  df_runs['best_val_mse'].std(),
           'pixacc_mean':  df_runs['pix_acc'].mean(),
           'pixacc_std':   df_runs['pix_acc'].std(),
           'n_runs': len(df_runs)}
pd.DataFrame([overall]).round(5).to_csv(os.path.join(RES_DIR, 'cv_overall.csv'), index=False)

print("\n── Per-fold (mean ± std over 3 seeds) ──")
print(per_fold.to_string())
print(f"\n── OVERALL over {overall['n_runs']} runs ──")
print(f"  Val MSE : {overall['val_mse_mean']:.5f} ± {overall['val_mse_std']:.5f}")
print(f"  Pix-acc : {overall['pixacc_mean']:.2f}% ± {overall['pixacc_std']:.2f}%")

# Best run = deployment model.
best_row = df_runs.loc[df_runs['best_val_mse'].idxmin()]
deploy_dst = os.path.join(SAVE_DIR, f'Unet_{MODEL_NAME}_deploy_best.pth')
torch.save(torch.load(best_row['ckpt'], map_location='cpu'), deploy_dst)
print(f"\nDeployment model = fold{int(best_row['fold'])} seed{int(best_row['seed'])} "
      f"(val {best_row['best_val_mse']:.5f}) → {deploy_dst}")

# CV loss curves (all 15 runs).
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for r in all_runs:
    ep = range(1, len(r['val_hist']) + 1)
    ax[0].plot(ep, r['train_hist'], alpha=0.5)
    ax[1].plot(ep, r['val_hist'], alpha=0.5, label=f"f{r['fold']}s{r['seed']}")
ax[0].set_title('Train MSE — all runs'); ax[0].set_xlabel('Epoch'); ax[0].grid(alpha=.3)
ax[1].set_title('Val MSE — all runs');   ax[1].set_xlabel('Epoch'); ax[1].grid(alpha=.3)
ax[1].legend(fontsize=6, ncol=3)
plt.suptitle(f'DITU-net {N_FOLDS}-Fold x {len(SEEDS)} seeds', fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, 'cv_loss_curves.png'), dpi=180)
plt.show()
print("✅ Cell 5 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Held-out TEST evaluation (image reconstruction, unbiased)
# ═══════════════════════════════════════════════════════════════
# Group-disjoint test set, never seen in CV. (Downstream WPCM RMSE vs real
# SCADA is done in CELLs 7–8.)

test_ds = Subset(full_ds, list(test_pos))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

model = UNet(3, 1, 32, DROPOUT).to(device)
model.load_state_dict(torch.load(deploy_dst, map_location=device)['model'])
model.eval()

crit, test_mse = nn.MSELoss(), 0.0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        test_mse += crit(model(xb), yb).item()
test_mse /= max(1, len(test_loader))

print(f"Held-out TEST MSE : {test_mse:.5f}  (pix-acc {rmse_to_pixacc(test_mse):.2f}%)")
pd.DataFrame([{'test_mse': test_mse, 'pix_acc': rmse_to_pixacc(test_mse)}]).round(5)\
  .to_csv(os.path.join(RES_DIR, 'test_set_eval.csv'), index=False)

def denorm(t): return (t * 0.5 + 0.5).clamp(0, 1)
xb, yb = next(iter(test_loader))
xb = xb[:8].to(device); yb = yb[:8].to(device)
with torch.no_grad(): pb = model(xb)
ncol = min(8, xb.shape[0])
fig, axes = plt.subplots(3, ncol, figsize=(3*ncol, 9))
for c in range(ncol):
    axes[0, c].imshow(denorm(xb[c]).cpu().permute(1, 2, 0).numpy()); axes[0, c].axis('off')
    axes[1, c].imshow(denorm(pb[c]).cpu().squeeze().numpy(), cmap='gray'); axes[1, c].axis('off')
    axes[2, c].imshow(denorm(yb[c]).cpu().squeeze().numpy(), cmap='gray'); axes[2, c].axis('off')
for r, lbl in enumerate(['Input scatter', 'Predicted WPC', 'Ground truth']):
    axes[r, 0].set_ylabel(lbl, fontsize=9)
plt.suptitle('Held-out test: Input → Predicted → GT', fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, 'test_grid.png'), dpi=150); plt.show()
print("✅ Cell 6 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Robust pixel-mapping + DKC  (base paper Sec. III-B, ported)
# ═══════════════════════════════════════════════════════════════
# Fixes the step/vertical-riser bug:
#   • NaN-mask empty/all-bright columns -> no spurious power=1.0 at edges
#   • mean-of-argmin (not first argmin) -> stable line centre
#   • monotone jump guard (<0.1)        -> rejects vertical pixel artefacts
#   • C0-continuous flats b1=f(x1), b2=f(x2) -> NO hard clamps -> NO steps
#   • robust bisection + Newton with retries (paper paramFunc)
#   • try/except fallback so batch inference never crashes

from numpy import polyfit, poly1d
from scipy import optimize


class _PolyF:
    """Polynomial with cheap analytic 1st/2nd/3rd derivatives (paper)."""
    def __init__(self, f):
        self.f, self.o, self.c = f, f.o, f.coefficients
    def df(self, x):
        return sum(x**(self.o-i-1) * self.c[i] * (self.o-i) for i in range(self.o))
    def ddf(self, x):
        return sum(x**(self.o-i-2) * self.c[i] * (self.o-i-1) * (self.o-i)
                   for i in range(self.o-1))
    def dddf(self, x):
        return sum(x**(self.o-i-3) * self.c[i] * (self.o-i-2)*(self.o-i-1)*(self.o-i)
                   for i in range(self.o-2))


def _find_zero(f, b, c=0.8):
    return optimize.bisect(lambda x: f(x) - b, 0, c)


def _param_func(f, r1=0.19):
    """Locate cut-in (x1) and rated (x2) stationary points (paper paramFunc)."""
    fx = _PolyF(f)
    # ---- cut-in ----
    ok, cnt, first, f_x = False, 0, True, True
    while not ok:
        cnt += 1
        try:
            if cnt > 20:
                if first: r1, first = 0.19, False
                x1 = optimize.newton(fx.df, r1, fprime2=fx.dddf)
            else:
                x1 = optimize.newton(fx.df, r1, fprime=fx.ddf, fprime2=fx.dddf)
            ok = True
            if f_x:
                if x1 > r1 or f(x1) > 0.01:
                    r1 -= 0.005; ok = False
                    if r1 < 0.08: f_x, r1 = False, 0.19
            else:
                if x1 > r1: r1 -= 0.005; ok = False
        except Exception:
            r1 -= 0.005
        if r1 < -0.5:
            x1, ok = 0.0, True
    b1 = float(f(x1))
    # ---- rated ----
    r2 = 0.45
    for thr in (0.95, 0.85, 0.75, 0.65):
        try: r2 = _find_zero(f, thr); break
        except Exception: pass
    ok = False
    while not ok:
        try:
            x2 = optimize.newton(fx.df, r2, fprime2=fx.dddf)
            ok = True
            if x2 < r2: r2 += 0.005; ok = False
        except Exception:
            r2 += 0.005
        if r2 > 1.5:
            x2, ok = 1.0, True
    b2 = float(f(x2))
    return x1, b1, x2, b2


class _FinalF:
    """Piecewise WPC: flat b1 | poly | flat b2 (C0 continuous, paper Eq.17)."""
    def __init__(self, f, x1, b1, x2, b2):
        self.f, self.x1, self.b1, self.x2, self.b2 = f, x1, b1, x2, b2
    def __call__(self, x):
        x = np.asarray(x, dtype=float)
        scalar = (x.ndim == 0)
        x = np.atleast_1d(x)
        y = self.f(x)
        y[x < self.x1] = self.b1
        y[x > self.x2] = self.b2
        y = np.clip(y, 0.0, 1.0)
        return float(y[0]) if scalar else y


def image_to_curve(pred_tensor, p=12):
    """
    pred_tensor : (1,256,256) or (256,256) torch tensor, normalised [-1,1].
    Returns callable wpc_fn(ws_norm) -> power_norm  (vectorised, clipped [0,1]).
    """
    img = ((pred_tensor.squeeze().detach().cpu().float() * 0.5) + 0.5).clamp(0, 1).numpy()
    img255 = img * 255.0
    crop = img255[31:228, 32:230]          # validated plot area (paper)
    H, W = crop.shape
    xL = np.arange(0, 1, 1.0 / W)[:W]
    y = np.full(W, np.nan)
    y0 = 0.0
    for i in range(W):
        col = crop[:, i].copy()
        vm = col.min()
        if vm > 220:                       # empty column -> no curve here
            y[i] = np.nan; continue
        y[i] = 1.0 - np.mean(np.where(col == vm)[0]) / H   # mean-of-argmin, inverted
        while y[i] - y0 > 0.1:             # reject implausible vertical jumps
            col[np.argmin(col)] = 224
            vm = col.min()
            if vm > 220:
                y[i] = np.nan; break
            y[i] = 1.0 - np.mean(np.where(col == vm)[0]) / H
        if not np.isnan(y[i]):
            y0 = y[i]

    mask = ~np.isnan(y)
    xv, yv = xL[mask], y[mask]
    if len(xv) < 8:                        # degenerate prediction -> safe fallback
        return lambda x: np.clip(np.asarray(x, float), 0, 1)
    xv[-1] = 1.0

    try:
        f3 = poly1d(polyfit(xv, yv, p))
        x1, b1, x2, b2 = _param_func(f3, 0.19)
        fxx = _FinalF(f3, x1, b1, x2, b2)
        if b1 < 0 and x1 > 0:              # re-anchor cut-in where curve hits 0
            x1 = _find_zero(fxx, 0, 0.2); b1 = fxx(x1)
            fxx = _FinalF(fxx, x1, b1, x2, b2)
        return fxx
    except Exception as e:
        print(f"    ⚠️ DKC fallback ({e})")
        f3 = poly1d(polyfit(xv, yv, p))
        return lambda x: np.clip(f3(np.asarray(x, float)), 0, 1)


# ── Load normalisation + deployment model for inference ──────────────────────
NORM_PATH = _resolve(NORM_CANDIDATES)
assert NORM_PATH is not None, "turbine_norms.json not found in any candidate path."
with open(NORM_PATH) as fh:
    saved_norms = json.load(fh)
print(f"Norms loaded: {NORM_PATH}  ({len(saved_norms)} turbines)")

infer_model = UNet(3, 1, 32, DROPOUT).to(device)
infer_model.load_state_dict(torch.load(deploy_dst, map_location=device)['model'])
infer_model.eval()
print(f"Inference model loaded: {deploy_dst}")


def build_inference_image(ws_n, val_n):
    """Render scatter exactly like the dataset generator → (1,3,256,256)."""
    n  = max(len(ws_n), 1)
    ms = float(np.clip(np.sqrt(3528.0 / n), 0.4, 5.0))   # paper Eq.24 density rule
    fig, ax = plt.subplots(figsize=(2.56, 2.56))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    ax.plot(ws_n, val_n, '.', color='black', markersize=ms)
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=100); plt.close(fig); buf.seek(0)
    pil = Image.open(buf).convert('RGB').resize((256, 256))
    t = transforms.Compose([transforms.ToTensor(),
                            transforms.Normalize((0.5,)*3, (0.5,)*3)])(pil)
    return t.unsqueeze(0)


print("✅ Cell 7 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Batch inference: all turbines × all months  (Scenario S1, raw)
# ═══════════════════════════════════════════════════════════════
# Model INPUT = INFER_SOURCE scatter (GP by default, matching training+deployment).
# Evaluation REFERENCE = TP 90th-percentile bins (independent of the model).

TURBINES    = [f'WTG{i}' for i in range(1, 21)]
TEST_YEARS  = None          # e.g. [2024, 2025] or None=all
TEST_MONTHS = None
OUT_DIR  = '/kaggle/working/batch_results'
GRID_DIR = f'{OUT_DIR}/monthly_grids'
OVLY_DIR = f'{OUT_DIR}/monthly_overlays'
for d in (OUT_DIR, GRID_DIR, OVLY_DIR): os.makedirs(d, exist_ok=True)

df_full = pd.read_csv(CONSOLIDATED_CSV)
df_full['time'] = pd.to_datetime(df_full['time'])
df_full['year_month'] = df_full['time'].dt.to_period('M')
m = pd.Series(True, index=df_full.index)
if TEST_YEARS  is not None: m &= df_full['time'].dt.year.isin(TEST_YEARS)
if TEST_MONTHS is not None: m &= df_full['time'].dt.month.isin(TEST_MONTHS)
df_full = df_full[m].copy()
all_months = sorted(df_full['year_month'].unique())
print(f"{len(all_months)} months × {len(TURBINES)} turbines | input={INFER_SOURCE}")

X_AXIS = np.linspace(0, 1, 300)
COLORS = cm.tab20(np.linspace(0, 1, 20))


def tp_ref(ws_n, tp_n, nb=25):
    e = np.linspace(0, 1, nb + 1); xs, ys = [], []
    for i in range(nb):
        s = (ws_n >= e[i]) & (ws_n < e[i+1])
        if s.sum() >= 5:
            xs.append((e[i] + e[i+1]) / 2); ys.append(float(np.percentile(tp_n[s], 90)))
    return np.array(xs), np.array(ys)


def wmape(pred, true):
    nz = true > 0.05
    return np.nan if nz.sum() < 3 else float(100*np.abs(pred[nz]-true[nz]).sum()/true[nz].sum())


def ss(pred, true, tol): return float(100*np.mean(np.abs(pred-true) <= tol))


def run_one(tid, ym):
    nm = saved_norms.get(tid)
    if nm is None: return None
    sub = df_full[(df_full['Device Name'] == tid) & (df_full['year_month'] == ym)]
    if len(sub) < 20: return None
    ws = sub['Avg Wind Speed(m/s)'].values
    gp = sub['Avg Grid active power(kW)'].values
    tp = sub['Avg Theoretical Active Power(kW)'].values
    ws_n = np.clip(ws / nm['WS_MAX'], 0, 1)
    gp_n = np.clip(np.maximum(gp, 0) / nm['P_MAX'], 0, 1)
    tp_n = np.clip(tp / nm['P_MAX'], 0, 1)
    v = ~(np.isnan(ws_n) | np.isnan(gp_n) | np.isnan(tp_n))
    ws_n, gp_n, tp_n = ws_n[v], gp_n[v], tp_n[v]
    if v.sum() < 20: return None

    src = tp_n if INFER_SOURCE == 'TP' else gp_n     # model input scatter
    try:
        xt = build_inference_image(ws_n, src).to(device)
        with torch.no_grad(): pred = infer_model(xt)[0]
        wpc_fn = image_to_curve(pred)
    except Exception as e:
        print(f"  ⚠️ {tid} {ym}: {e}"); return None

    xr, yr = tp_ref(ws_n, tp_n)                       # evaluation reference (TP)
    if len(xr) < 5: return None
    wp   = np.array([float(wpc_fn(x)) for x in xr])
    rmse = float(np.sqrt(np.mean((wp - yr)**2)))
    mae  = float(np.mean(np.abs(wp - yr)))
    nz   = yr > 0.05
    mape = float(100*np.mean(np.abs((wp[nz]-yr[nz])/yr[nz]))) if nz.sum() else np.nan
    return {'turbine': tid, 'month': str(ym), 'n': int(v.sum()),
            'RMSE': round(rmse,5), 'MAE': round(mae,5), 'MAPE': round(mape,2),
            'WMAPE': round(wmape(wp, yr),2),
            'SS_05': round(ss(wp,yr,0.05),2), 'SS_10': round(ss(wp,yr,0.10),2),
            'SS_15': round(ss(wp,yr,0.15),2),
            '_fn': wpc_fn, '_ws': ws_n, '_src': src, '_xr': xr, '_yr': yr}


def plot_grid(ym, R):
    nc = 4; nr = int(np.ceil(len(R)/nc))
    fig, ax = plt.subplots(nr, nc, figsize=(nc*4.2, nr*3.4), constrained_layout=True)
    af = np.atleast_1d(ax).flatten()
    for j, r in enumerate(R):
        wpc = np.array([float(r['_fn'](x)) for x in X_AXIS])
        af[j].scatter(r['_ws'], r['_src'], s=0.8, alpha=.15, color='steelblue', rasterized=True)
        af[j].scatter(r['_xr'], r['_yr'], s=18, color='green', zorder=5)
        af[j].plot(X_AXIS, wpc, 'r-', lw=1.8, label=f"RMSE={r['RMSE']:.4f}")
        af[j].set_xlim(0,1); af[j].set_ylim(-.05,1.1)
        af[j].set_title(f"{r['turbine']} n={r['n']}", fontsize=9)
        af[j].legend(fontsize=6, loc='upper left'); af[j].grid(alpha=.3)
    for j in range(len(R), len(af)): af[j].axis('off')
    fig.suptitle(f'DITU-net WPC — {ym} (input={INFER_SOURCE})',
                 fontsize=13, fontweight='bold', y=1.01)
    p = f'{GRID_DIR}/grid_{ym}.png'; fig.savefig(p, dpi=150, bbox_inches='tight')
    plt.close(fig); return p


def plot_overlay(ym, R):
    fig, ax = plt.subplots(figsize=(10, 6))
    for j, r in enumerate(R):
        ax.plot(X_AXIS, [float(r['_fn'](x)) for x in X_AXIS],
                lw=1.6, color=COLORS[j % 20], label=r['turbine'], alpha=.85)
    ax.set_xlabel('Normalised Wind Speed'); ax.set_ylabel('Normalised Power')
    ax.set_title(f'DITU-net WPC — All Turbines — {ym}', fontweight='bold')
    ax.set_xlim(0,1); ax.set_ylim(-.05,1.1); ax.grid(alpha=.3)
    ax.legend(fontsize=7, ncol=5, loc='upper left', framealpha=.7)
    plt.tight_layout(); p = f'{OVLY_DIR}/overlay_{ym}.png'
    fig.savefig(p, dpi=150); plt.close(fig); return p


rows, done, total = [], 0, len(all_months)*len(TURBINES)
for ym in all_months:
    ys, R = str(ym), []
    for tid in TURBINES:
        done += 1
        r = run_one(tid, ym)
        if r:
            R.append(r); rows.append({k: v for k, v in r.items() if not k.startswith('_')})
    if R:
        plot_grid(ys, R); plot_overlay(ys, R)
        print(f"  {ys}: {len(R)} turbines  [{done}/{total}]")
    del R; gc.collect()

df_res = pd.DataFrame(rows)
df_res.to_csv(f'{OUT_DIR}/all_results_raw.csv', index=False)

# Paper-style pivot tables (month × turbine).
for met in ['RMSE','MAE','MAPE','WMAPE','SS_05','SS_10','SS_15']:
    piv = df_res.pivot(index='month', columns='turbine', values=met).reindex(columns=TURBINES)
    piv['MEAN'], piv['MEDIAN'] = piv.mean(1), piv.median(1)
    piv.loc['MEAN'], piv.loc['MEDIAN'] = piv.mean(0), piv.median(0)
    piv.to_csv(f'{OUT_DIR}/table_{met}.csv')

# Monthly summary (mean±std over turbines).
summary = df_res.groupby('month').agg(
    RMSE_mean=('RMSE','mean'), RMSE_std=('RMSE','std'),
    MAE_mean=('MAE','mean'), MAPE_mean=('MAPE','mean'), WMAPE_mean=('WMAPE','mean'),
    SS05=('SS_05','mean'), SS10=('SS_10','mean'), SS15=('SS_15','mean'),
    n=('turbine','count')).round(4)
summary.to_csv(f'{OUT_DIR}/summary_by_month.csv')

# Overall headline numbers.
overall_m = df_res[['RMSE','MAE','MAPE','WMAPE','SS_05','SS_10','SS_15']]\
              .agg(['mean','std','min','max']).round(4)
overall_m.to_csv(f'{OUT_DIR}/overall_metrics.csv')
print("\n=== OVERALL (all turbine-months, S1 raw) ===")
print(overall_m.to_string())

# RMSE heatmap.
piv = df_res.pivot(index='month', columns='turbine', values='RMSE').reindex(columns=TURBINES)
fig, ax = plt.subplots(figsize=(20, max(5, len(piv)*0.55)))
im = ax.imshow(piv.values.astype(float), aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=0.10)
ax.set_xticks(range(len(TURBINES))); ax.set_xticklabels(TURBINES, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index.tolist(), fontsize=9)
ax.set_title('DITU-net RMSE(TP) — month × turbine', fontweight='bold', pad=10)
plt.colorbar(im, ax=ax, label='RMSE(TP)', shrink=.8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/rmse_heatmap.png', dpi=180, bbox_inches='tight'); plt.show()
print(f"\n✅ Cell 8 done — outputs in {OUT_DIR}/")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8b — Scenario preprocessing S1 / S2 / S3   (paper-matched)
# ═══════════════════════════════════════════════════════════════
# Only the INPUT CLEANING changes between scenarios — model, marker-size,
# extractor and the TP-90th reference are IDENTICAL -> apples-to-apples with
# the base paper (Table IV S1/S2/S3).
#   S1 raw     : no cleaning (online use, what we deploy)
#   S2 rough   : drop points far below the local TP band (quick low-latency clean)
#   S3 careful : S2 + per-WS-bin IQR outlier removal (offline expert-style clean)
#
# Cleaning is applied to the (ws, GP) points the MODEL SEES. The scoring
# reference is always built from the full TP column (unchanged), exactly as the
# paper evaluates against its independent test points.

TEST_YEAR_ONLY = 2025      # for now we evaluate on 2025 only (future: ACT data)

def preprocess_points(ws_n, src_n, tp_n, scenario):
    """Return cleaned (ws, src) for the scenario. tp_n is guidance only."""
    if scenario == 'S1':
        return ws_n, src_n
    keep = np.ones(len(ws_n), dtype=bool)
    if scenario in ('S2', 'S3'):
        # Rough filter: above cut-in, drop points < 50% of local TP capability.
        above = ws_n > 0.30
        ratio = src_n / (tp_n + 1e-6)
        keep &= ~(above & (ratio < 0.50))
    if scenario == 'S3':
        # Careful filter: per-WS-bin IQR removal on the survivors.
        edges = np.linspace(0, 1, 26)
        for i in range(25):
            sel = keep & (ws_n >= edges[i]) & (ws_n < edges[i+1])
            if sel.sum() < 8:
                continue
            v = src_n[sel]
            q1, q3 = np.percentile(v, 25), np.percentile(v, 75)
            iqr = q3 - q1
            lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
            keep &= ~(sel & ((src_n < lo) | (src_n > hi)))
    if keep.sum() < 20:          # never delete everything
        return ws_n, src_n
    return ws_n[keep], src_n[keep]

print(f"✅ Cell 8b — scenario filters ready (test year = {TEST_YEAR_ONLY})")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8c — Batch inference for S1/S2/S3 (2025 only) + paper comparison
# ═══════════════════════════════════════════════════════════════
SCEN_DIR = '/kaggle/working/scenario_results'
os.makedirs(SCEN_DIR, exist_ok=True)

df_year = df_full[df_full['time'].dt.year == TEST_YEAR_ONLY].copy()
months_year = sorted(df_year['year_month'].unique())
print(f"Testing {len(months_year)} months of {TEST_YEAR_ONLY}: "
      f"{months_year[0]} → {months_year[-1]}  | input={INFER_SOURCE}")

def run_one_scenario(tid, ym, scenario):
    nm = saved_norms.get(tid)
    if nm is None: return None
    sub = df_year[(df_year['Device Name'] == tid) & (df_year['year_month'] == ym)]
    if len(sub) < 20: return None
    ws = sub['Avg Wind Speed(m/s)'].values
    gp = sub['Avg Grid active power(kW)'].values
    tp = sub['Avg Theoretical Active Power(kW)'].values
    ws_n = np.clip(ws / nm['WS_MAX'], 0, 1)
    gp_n = np.clip(np.maximum(gp, 0) / nm['P_MAX'], 0, 1)
    tp_n = np.clip(tp / nm['P_MAX'], 0, 1)
    v = ~(np.isnan(ws_n) | np.isnan(gp_n) | np.isnan(tp_n))
    ws_n, gp_n, tp_n = ws_n[v], gp_n[v], tp_n[v]
    if v.sum() < 20: return None

    src_full = gp_n if INFER_SOURCE == 'GP' else tp_n        # model input (GP default)
    ws_c, src_c = preprocess_points(ws_n, src_full, tp_n, scenario)
    try:
        xt = build_inference_image(ws_c, src_c).to(device)
        with torch.no_grad(): pred = infer_model(xt)[0]
        wpc_fn = image_to_curve(pred)
    except Exception as e:
        print(f"  ⚠️ {tid} {ym} {scenario}: {e}"); return None

    xr, yr = tp_ref(ws_n, tp_n)                              # reference = full TP (unchanged)
    if len(xr) < 5: return None
    wp   = np.array([float(wpc_fn(x)) for x in xr])
    rmse = float(np.sqrt(np.mean((wp - yr)**2)))
    mae  = float(np.mean(np.abs(wp - yr)))
    nz   = yr > 0.05
    mape = float(100*np.mean(np.abs((wp[nz]-yr[nz])/yr[nz]))) if nz.sum() else np.nan
    return {'scenario': scenario, 'turbine': tid, 'month': str(ym), 'n': int(v.sum()),
            'RMSE': round(rmse,5), 'MAE': round(mae,5), 'MAPE': round(mape,2),
            'WMAPE': round(wmape(wp, yr),2),
            'SS_05': round(ss(wp,yr,0.05),2), 'SS_10': round(ss(wp,yr,0.10),2),
            'SS_15': round(ss(wp,yr,0.15),2)}

scen_rows = []
for scenario in ('S1', 'S2', 'S3'):
    print(f"\n=== Running {scenario} ({TEST_YEAR_ONLY}) ===")
    for ym in months_year:
        for tid in TURBINES:
            r = run_one_scenario(tid, ym, scenario)
            if r: scen_rows.append(r)
    print(f"  {scenario}: {sum(1 for r in scen_rows if r['scenario']==scenario)} turbine-months")

df_scen = pd.DataFrame(scen_rows)
df_scen.to_csv(f'{SCEN_DIR}/all_scenarios_raw_{TEST_YEAR_ONLY}.csv', index=False)

ours_by_scen = df_scen.groupby('scenario')[
    ['RMSE','MAE','MAPE','WMAPE','SS_05','SS_10','SS_15']].mean().round(5)
ours_by_scen.to_csv(f'{SCEN_DIR}/ours_by_scenario_{TEST_YEAR_ONLY}.csv')
print("\n── Ours (mean over turbine-months), by scenario ──")
print(ours_by_scen.to_string())

# Base-paper DITU-net numbers (Table IV) for S1/S2/S3.
PAPER = {
 'S1': {'RMSE':0.06960,'MAE':0.04688,'MAPE':12.80,'WMAPE':13.36,'SS_05':67.91,'SS_10':86.26,'SS_15':93.91},
 'S2': {'RMSE':0.06960,'MAE':0.04684,'MAPE':12.82,'WMAPE':13.35,'SS_05':67.85,'SS_10':86.25,'SS_15':93.90},
 'S3': {'RMSE':0.06893,'MAE':0.04616,'MAPE':12.68,'WMAPE':13.14,'SS_05':68.34,'SS_10':86.50,'SS_15':93.91},
}

def _better(metric, o, p):
    if metric.startswith('SS'): return '✅' if o >= p else '↓'
    return '✅' if o <= p else '↑'

cmp_rows = []
for sc in ('S1','S2','S3'):
    if sc not in ours_by_scen.index: continue
    for met in ['RMSE','MAE','MAPE','WMAPE','SS_05','SS_10','SS_15']:
        o, p = float(ours_by_scen.loc[sc, met]), PAPER[sc][met]
        cmp_rows.append({'scenario':sc,'metric':met,'ours':round(o,5),
                         'paper':p,'verdict':_better(met,o,p)})
df_cmp = pd.DataFrame(cmp_rows)
df_cmp.to_csv(f'{SCEN_DIR}/comparison_all_scenarios_{TEST_YEAR_ONLY}.csv', index=False)
print("\n── Ours vs Base paper (all scenarios) ──")
print(df_cmp.to_string(index=False))
print("\nRMSE/MAE/MAPE/WMAPE → lower better;  SS_* → higher better.")
print("✅ Cell 8c done")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — BONUS: seed sweep 42..45 on ONE fixed split
# ═══════════════════════════════════════════════════════════════
# Isolates INIT variance only (split frozen = fold 0). Reported separately from
# the K-Fold result because it answers a different question (retrain stability).

SEED_SWEEP = [42, 43, 44, 45]
sweep_tr, sweep_va = folds[0]

sweep_rows = []
for s in SEED_SWEEP:
    print(f"▶ Seed sweep | seed {s}")
    r = train_one_run(sweep_tr, sweep_va, seed=s, fold='sweep', verbose=True)
    sweep_rows.append({'seed': s, 'best_val_mse': r['best_val_mse'],
                       'pix_acc': r['pix_acc'], 'best_epoch': r['best_epoch']})

df_sweep = pd.DataFrame(sweep_rows)
df_sweep.to_csv(os.path.join(RES_DIR, 'seed_sweep_42_45.csv'), index=False)
print("\n── Seed sweep (fixed split, init variance only) ──")
print(df_sweep.to_string(index=False))
print(f"  Val MSE : {df_sweep['best_val_mse'].mean():.5f} "
      f"± {df_sweep['best_val_mse'].std():.5f}")
print("✅ Cell 9 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — Side-by-side comparison with base paper (Scenario S1)
# ═══════════════════════════════════════════════════════════════
# Base-paper DITU-net numbers from Table IV (S1 column).
# Ours = mean over all turbine-months from CELL 8 (df_res).

PAPER_DITU_S1 = {
    'RMSE': 0.06960, 'MAE': 0.04688, 'MAPE': 12.80, 'WMAPE': 13.36,
    'SS_05': 67.91, 'SS_10': 86.26, 'SS_15': 93.91,
}
ours = df_res[['RMSE','MAE','MAPE','WMAPE','SS_05','SS_10','SS_15']].mean()

def _better(metric, ours_v, paper_v):
    if metric.startswith('SS'):
        return '✅ better' if ours_v >= paper_v else '↓ lower'
    return '✅ better' if ours_v <= paper_v else '↑ higher'

cmp = pd.DataFrame({
    'Metric'      : list(PAPER_DITU_S1.keys()),
    'Base paper'  : [PAPER_DITU_S1[k] for k in PAPER_DITU_S1],
    'Ours (mean)' : [round(float(ours[k]), 5) for k in PAPER_DITU_S1],
})
cmp['Verdict'] = [_better(mt, o, p) for mt, o, p in
                  zip(cmp['Metric'], cmp['Ours (mean)'], cmp['Base paper'])]
cmp.to_csv(f'{OUT_DIR}/comparison_vs_basepaper_S1.csv', index=False)
print("=== DITU-net S1: Ours vs Base Paper (Table IV) ===")
print(cmp.to_string(index=False))
print("\nNote: RMSE/MAE/MAPE/WMAPE → lower is better; SS_05/10/15 → higher is better.")
print("✅ Cell 10 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 11 — Reproducibility manifest + methodology statement
# ═══════════════════════════════════════════════════════════════
manifest = {
    'design'          : 'X=GP (actual) scatter / Y=TP-fitted S-curve',
    'infer_source'    : INFER_SOURCE,
    'model_name'      : MODEL_NAME,
    'dataset_len'     : int(DATASET_LEN),
    'group_mode'      : GROUP_MODE,
    'split': {
        'test_fraction': TEST_FRACTION, 'split_seed': SPLIT_SEED,
        'n_folds': N_FOLDS, 'seeds_per_fold': SEEDS,
        'total_cv_runs': N_FOLDS * len(SEEDS), 'grouping': 'turbine-month (GroupKFold)',
    },
    'hyperparams': {
        'batch_size': BATCH_SIZE, 'epochs': NUM_EPOCHS, 'lr': LR,
        'dropout': DROPOUT, 'weight_decay': WEIGHT_DECAY,
        'patience': PATIENCE, 'lr_patience': LR_PATIENCE,
    },
    'inference': {
        'marker_size_rule': 'sqrt(3528/n) == paper Eq.24 (K=0.07, MS_DIT=6)',
        'eval_reference'  : 'TP 90th-percentile, 25 bins',
        'extractor'       : 'pixel-map + poly(deg12) + DKC (paper Sec.III-B)',
    },
    'versions': {'torch': torch.__version__, 'numpy': np.__version__},
    'norm_path': NORM_PATH, 'group_path': GROUP_PATH, 'deploy_checkpoint': deploy_dst,
}
with open(os.path.join(SAVE_DIR, 'run_manifest.json'), 'w') as fh:
    json.dump(manifest, fh, indent=2)
print(json.dumps(manifest, indent=2))

METHOD_STATEMENT = """
Evaluation protocol (Scenario S1, raw SCADA):
Training pairs use the ACTUAL grid-power (GP) scatter as input X and a clean
TP-fitted S-curve as target Y, so the network learns to recover the neat wind
power curve (true capability) from messy actual output — the deployment task.
Robustness to curtailment/outliers is obtained via heavy synthetic
curtailment-shaped diversity in training, the base paper's strategy on real data. For each turbine-month, raw 10-min records are normalised
per turbine (WS_MAX, P_MAX = 99.5th percentile) and rendered as a 256x256 scatter
using the base paper's marker-size rule (Eq. 24, K=0.07, MS_DIT=6). The trained
DITU-net receives ONLY this image and outputs a neat-WPC image, converted to a
parametric curve via the paper's pixel-mapping + polynomial fit +
Domain-Knowledge-Correction (Sec. III-B). Accuracy is measured against an
independent reference: the 90th-percentile of theoretical active power binned
over 25 wind-speed intervals. The model never observes this reference, so no
curve is fit to the evaluation target. Metrics (RMSE, MAE, MAPE, WMAPE,
alpha-SS at 0.05/0.10/0.15) follow the base paper. Model selection used 5-fold
GROUP cross-validation (grouped by turbine-month, no augmentation leakage) x 3
seeds (15 runs) on an 85% dev set; a fixed, group-disjoint 15% test set was held
out for the final unbiased estimate.
"""
with open(os.path.join(RES_DIR, 'methodology_statement.txt'), 'w') as fh:
    fh.write(METHOD_STATEMENT)
print(METHOD_STATEMENT)
print("✅ Cell 11 done — pipeline complete")